In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

sns.set_theme(style="darkgrid")

with open("portfolio_data.pkl", "rb") as f:
    data = pickle.load(f)

# Unpack the objects you'll use throughout this notebook
portfolio_returns = data["portfolio_returns"]   # pd.Series of daily log returns
log_returns       = data["log_returns"]         # pd.DataFrame, individual assets
weights           = data["weights"]             # np.ndarray
cov_matrix        = data["cov_matrix"]          # annualised, pd.DataFrame
conf_levels       = data["conf_levels"]         # [0.95, 0.99]
lookback          = data["lookback"]            # 252

print("Loaded successfully.")
print(f"Portfolio returns shape : {portfolio_returns.shape}")
print(f"Date range: {portfolio_returns.index[0].date()} → {portfolio_returns.index[-1].date()}")
print(f"Confidence levels: {conf_levels}")

Loaded successfully.
Portfolio returns shape : (2514,)
Date range: 2015-01-05 → 2024-12-30
Confidence levels: [0.95, 0.99]


In [ ]:
import os

_cwd = os.getcwd()
_proj_root = _cwd if os.path.exists(os.path.join(_cwd, "README.md")) else os.path.dirname(_cwd)
RESULTS = os.path.join(_proj_root, "results", "phase_2_var")
os.makedirs(RESULTS, exist_ok=True)
print(f"Results folder: {RESULTS}")

In [2]:
# VaR(c, h) = the loss L such that P(loss > L) = 1 - c
# over horizon h (here h = 1 day)
#
# At 95% confidence: we expect to EXCEED this loss only 5% of trading days
# At 99% confidence: only 1% of trading days
#
# Three methods differ in HOW they estimate that loss threshold:
#   1. Historical Simulation  → use the empirical distribution directly
#   2. Parametric             → assume normality, use μ and σ
#   3. Monte Carlo            → simulate 10,000 paths with correlated shocks
#
# All three should give similar answers under normal conditions.
# They diverge in the tails — that divergence IS the story.

# We'll compute point-in-time VaR first (full sample),
# then build a rolling engine in Phase 4 (backtesting).

# Use the FULL sample for point-in-time estimates
returns = portfolio_returns.values   # convert to numpy array for speed
print(f"Total return observations: {len(returns)}")

Total return observations: 2514


In [3]:
# Idea: sort the past 252 days of returns worst-to-best.
#       The 5th percentile IS your 95% VaR. No distribution assumed.
#
# VaR_HS(c) = -percentile(returns, 1-c)
# The negative sign converts a return (negative number) to a loss (positive number)

def historical_var(returns: np.ndarray, confidence: float) -> float:
    """
    Historical simulation VaR.
    returns    : 1D array of portfolio log returns
    confidence : e.g. 0.95 or 0.99
    Returns VaR as a POSITIVE number (loss convention)
    """
    return -np.percentile(returns, (1 - confidence) * 100)


hs_var = {c: historical_var(returns, c) for c in conf_levels}

print("── Historical Simulation VaR ────────────────────────────")
for c, v in hs_var.items():
    print(f"  {int(c*100)}% VaR : {v*100:.4f}%  →  on a $1M portfolio = ${v*1e6:,.0f}")

── Historical Simulation VaR ────────────────────────────
  95% VaR : 1.0610%  →  on a $1M portfolio = $10,610
  99% VaR : 1.8886%  →  on a $1M portfolio = $18,886


In [ ]:
plt.figure(figsize=(12, 5))
plt.hist(returns * 100, bins=80, density=True,
         color="steelblue", alpha=0.6, label="Empirical returns")

for c, v in hs_var.items():
    plt.axvline(-v * 100, color="red" if c == 0.99 else "orange",
                linewidth=2, linestyle="--",
                label=f"HS VaR {int(c*100)}% = {v*100:.2f}%")

plt.title("Historical Simulation VaR — Return Distribution")
plt.xlabel("Daily Return (%)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, "01_hs_var_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

In [5]:
# Assumes returns ~ Normal(μ, σ²)
# VaR_param(c) = -(μ + z_c * σ)
# where z_c = norm.ppf(1 - c)   ← negative number (left tail quantile)
#
# For a portfolio:
#   σ_p = sqrt(w^T * Σ_daily * w)
#   Σ_daily = Σ_annual / 252
#
# NOTE: μ is often set to 0 for short-horizon VaR (daily mean is negligible
#       relative to daily vol — conservative and standard practice).

def parametric_var(
    weights: np.ndarray,
    cov_matrix: pd.DataFrame,
    confidence: float,
    mu: float = 0.0,       # set to 0 by default (conservative)
) -> float:
    """
    Parametric (variance-covariance) VaR.
    Returns VaR as a POSITIVE number (loss convention).
    """
    cov_daily  = cov_matrix.values / 252          # annualised → daily
    port_var   = weights @ cov_daily @ weights     # scalar: portfolio daily variance
    port_sigma = np.sqrt(port_var)                 # portfolio daily std dev
    z          = norm.ppf(1 - confidence)          # e.g. -1.645 at 95%
    var        = -(mu + z * port_sigma)
    return var


param_var = {c: parametric_var(weights, cov_matrix, c) for c in conf_levels}

print("── Parametric (Variance-Covariance) VaR ────────────────")
for c, v in param_var.items():
    print(f"  {int(c*100)}% VaR : {v*100:.4f}%  →  on a $1M portfolio = ${v*1e6:,.0f}")

# Intermediate values — useful to show understanding in interviews
cov_daily  = cov_matrix.values / 252
port_sigma = np.sqrt(weights @ cov_daily @ weights)
print(f"\n  Daily portfolio σ : {port_sigma*100:.4f}%")
print(f"  z at 95%          : {norm.ppf(0.05):.4f}")
print(f"  z at 99%          : {norm.ppf(0.01):.4f}")

── Parametric (Variance-Covariance) VaR ────────────────
  95% VaR : 1.2132%  →  on a $1M portfolio = $12,132
  99% VaR : 1.7159%  →  on a $1M portfolio = $17,159

  Daily portfolio σ : 0.7376%
  z at 95%          : -1.6449
  z at 99%          : -2.3263


In [ ]:
mu_p    = portfolio_returns.mean()
sigma_p = portfolio_returns.std()
x       = np.linspace(portfolio_returns.min(), portfolio_returns.max(), 400)

plt.figure(figsize=(12, 5))
plt.hist(returns * 100, bins=80, density=True,
         color="steelblue", alpha=0.5, label="Empirical returns")
plt.plot(x * 100, norm.pdf(x, mu_p, sigma_p) / 100,
         color="black", linewidth=2, label="Normal fit")

for c, v in param_var.items():
    plt.axvline(-v * 100, color="red" if c == 0.99 else "orange",
                linewidth=2, linestyle="-.",
                label=f"Param VaR {int(c*100)}% = {v*100:.2f}%")

plt.title("Parametric VaR — Normal Distribution Assumption")
plt.xlabel("Daily Return (%)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, "02_parametric_var_normal.png"), dpi=150, bbox_inches="tight")
plt.show()

In [7]:
# Idea: simulate 10,000 possible "next day" portfolio returns
#       using the SAME covariance structure as the real data.
#       Then take the percentile of simulated losses.
#
# Steps:
#   1. Decompose Σ_daily via Cholesky: Σ = L L^T
#   2. Draw Z ~ N(0, I)   shape (N_sims, N_assets)
#   3. Correlated shocks  = Z @ L^T  (shape: N_sims × N_assets)
#   4. Simulated portfolio return = correlated_shocks @ weights
#   5. VaR = -percentile(simulated_portfolio_returns, 1-c)

N_SIMS   = 10_000
N_ASSETS = len(weights)

np.random.seed(42)   # reproducibility

# Step 1: Cholesky decomposition of the DAILY covariance matrix
cov_daily = cov_matrix.values / 252
L         = np.linalg.cholesky(cov_daily)   # lower triangular, shape (N_assets, N_assets)

print(f"Cov matrix shape : {cov_daily.shape}")
print(f"Cholesky L shape : {L.shape}")
print(f"\nCholesky L (first 3×3):\n{L[:3, :3].round(6)}")

# Verify: L @ L.T should equal cov_daily
recon_error = np.max(np.abs(L @ L.T - cov_daily))
print(f"\nMax reconstruction error (L @ L.T - Σ): {recon_error:.2e}  (should be ~0)")

Cov matrix shape : (8, 8)
Cholesky L shape : (8, 8)

Cholesky L (first 3×3):
[[ 0.011139  0.        0.      ]
 [ 0.012812  0.005082  0.      ]
 [ 0.009371 -0.001117  0.005553]]

Max reconstruction error (L @ L.T - Σ): 2.71e-20  (should be ~0)


In [8]:
# Step 2: Draw independent standard normal shocks
Z = np.random.standard_normal((N_SIMS, N_ASSETS))
print(f"Z shape: {Z.shape}")   # (10000, 8)

# Step 3: Introduce correlation via Cholesky
# Each row of Z_corr is a simulated daily return vector across all assets
Z_corr = Z @ L.T
print(f"Z_corr shape: {Z_corr.shape}")   # (10000, 8)

# Step 4: Simulated portfolio returns (dot with weights)
sim_portfolio_returns = Z_corr @ weights
print(f"Simulated portfolio returns shape: {sim_portfolio_returns.shape}")   # (10000,)

print(f"\nSimulated return stats:")
print(f"  Mean  : {sim_portfolio_returns.mean()*100:.4f}%")
print(f"  Std   : {sim_portfolio_returns.std()*100:.4f}%")
print(f"  Min   : {sim_portfolio_returns.min()*100:.4f}%")
print(f"  Max   : {sim_portfolio_returns.max()*100:.4f}%")

Z shape: (10000, 8)
Z_corr shape: (10000, 8)
Simulated portfolio returns shape: (10000,)

Simulated return stats:
  Mean  : 0.0078%
  Std   : 0.7339%
  Min   : -2.9518%
  Max   : 2.6263%


In [9]:
def montecarlo_var(
    sim_returns: np.ndarray,
    confidence: float,
) -> float:
    """
    Monte Carlo VaR from simulated portfolio return distribution.
    Returns VaR as a POSITIVE number (loss convention).
    """
    return -np.percentile(sim_returns, (1 - confidence) * 100)


mc_var = {c: montecarlo_var(sim_portfolio_returns, c) for c in conf_levels}

print("── Monte Carlo VaR (N=10,000 simulations) ───────────────")
for c, v in mc_var.items():
    print(f"  {int(c*100)}% VaR : {v*100:.4f}%  →  on a $1M portfolio = ${v*1e6:,.0f}")

── Monte Carlo VaR (N=10,000 simulations) ───────────────
  95% VaR : 1.1891%  →  on a $1M portfolio = $11,891
  99% VaR : 1.7007%  →  on a $1M portfolio = $17,007


In [ ]:
plt.figure(figsize=(12, 5))
plt.hist(sim_portfolio_returns * 100, bins=100, density=True,
         color="mediumpurple", alpha=0.6, label=f"MC Simulated ({N_SIMS:,} paths)")

for c, v in mc_var.items():
    plt.axvline(-v * 100, color="red" if c == 0.99 else "orange",
                linewidth=2, linestyle="--",
                label=f"MC VaR {int(c*100)}% = {v*100:.2f}%")

plt.title("Monte Carlo VaR — Simulated Return Distribution")
plt.xlabel("Simulated Daily Return (%)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, "03_montecarlo_var_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

In [11]:
comparison = pd.DataFrame({
    "Historical Sim (%)":  {c: round(hs_var[c]*100, 4)    for c in conf_levels},
    "Parametric (%)":      {c: round(param_var[c]*100, 4)  for c in conf_levels},
    "Monte Carlo (%)":     {c: round(mc_var[c]*100, 4)     for c in conf_levels},
})
comparison.index = [f"{int(c*100)}% VaR" for c in conf_levels]

print("── VaR Comparison Table ─────────────────────────────────")
print(comparison.to_string())

# Key things to notice:
# 1. Parametric VaR < HS VaR at 99% → normality underestimates fat tails
# 2. MC VaR ≈ Parametric VaR → both assume normal shocks in this implementation
# 3. The gap between Parametric and HS is your model risk in the tail

── VaR Comparison Table ─────────────────────────────────
         Historical Sim (%)  Parametric (%)  Monte Carlo (%)
95% VaR              1.0610          1.2132           1.1891
99% VaR              1.8886          1.7159           1.7007


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
methods = ["Historical Sim (%)", "Parametric (%)", "Monte Carlo (%)"]
colors  = ["steelblue", "tomato", "mediumpurple"]

for ax, c in zip(axes, conf_levels):
    vals = [comparison.loc[f"{int(c*100)}% VaR", m] for m in methods]
    bars = ax.bar(["Historical\nSim", "Parametric", "Monte Carlo"],
                  vals, color=colors, alpha=0.8, width=0.5)
    ax.bar_label(bars, fmt="%.3f%%", padding=3, fontsize=9)
    ax.set_title(f"{int(c*100)}% Confidence VaR Comparison")
    ax.set_ylabel("VaR (% of portfolio value)")
    ax.set_ylim(0, max(vals) * 1.3)

plt.suptitle("VaR Method Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, "04_var_method_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

In [13]:
PORTFOLIO_VALUE = 1_000_000

dollar_var = pd.DataFrame({
    "Historical Sim ($)": {c: hs_var[c]    * PORTFOLIO_VALUE for c in conf_levels},
    "Parametric ($)":     {c: param_var[c] * PORTFOLIO_VALUE for c in conf_levels},
    "Monte Carlo ($)":    {c: mc_var[c]    * PORTFOLIO_VALUE for c in conf_levels},
})
dollar_var.index = [f"{int(c*100)}% VaR" for c in conf_levels]

print(f"── Dollar VaR (Portfolio Value = ${PORTFOLIO_VALUE:,}) ──────────────")
print(dollar_var.map(lambda x: f"${x:,.0f}").to_string())

── Dollar VaR (Portfolio Value = $1,000,000) ──────────────
        Historical Sim ($) Parametric ($) Monte Carlo ($)
95% VaR            $10,610        $12,132         $11,891
99% VaR            $18,886        $17,159         $17,007


In [14]:
var_results = {
    "hs_var":                hs_var,
    "param_var":             param_var,
    "mc_var":                mc_var,
    "sim_portfolio_returns": sim_portfolio_returns,   # MC paths needed for CVaR
    "comparison_table":      comparison,
}

# Merge with phase 1 data and save
data.update(var_results)

with open("portfolio_data.pkl", "wb") as f:
    pickle.dump(data, f)

print("portfolio_data.pkl updated with Phase 2 VaR results.")
print("\nKeys now available:")
for k in data.keys():
    print(f"  {k}")

portfolio_data.pkl updated with Phase 2 VaR results.

Keys now available:
  prices
  log_returns
  portfolio_returns
  weights
  tickers
  stats
  cov_matrix
  corr_matrix
  lookback
  conf_levels
  hs_var
  param_var
  mc_var
  sim_portfolio_returns
  comparison_table
